# Analyze SINDy Autoencoder - Reaction-Diffusion

Loads a trained checkpoint and checks reconstruction / SINDy error, matching the original `analyze_rd_model*.ipynb` notebooks.

## Data caveat

**`reaction_diffusion.mat` must be present in this folder.** It is generated by the MATLAB solver in `../../../SindyAutoencoders/rd_solver/` and is not part of either repo. This notebook is a faithful port of the original `train_reactiondiffusion.ipynb` / `analyze_rd_model*.ipynb`, but it has only been verified with synthetic data of the right shape - not run end-to-end with the real reaction-diffusion data.

In [ ]:
import os, sys

# Path to the SindyAutoencoders_Torch folder (this repo). If you uploaded/cloned
# it into Colab's local disk or mounted it from Google Drive, set this to that
# location - e.g. '/content/SindyAutoencoders_Torch' or
# '/content/drive/MyDrive/SindyAutoencoders_Torch'.
REPO_ROOT = '/content/SindyAutoencoders_Torch'

if not os.path.isdir(REPO_ROOT):
    # fall back to running locally, from within this notebook's own example folder
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

EXAMPLE_DIR = os.path.join(REPO_ROOT, 'examples', 'rd')
os.chdir(EXAMPLE_DIR)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, EXAMPLE_DIR)
print('Working directory:', os.getcwd())


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from data import get_rd_data
from model import SindyAutoencoder
from sindy_utils import sindy_simulate


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('No GPU detected - in Colab, check Runtime > Change runtime type > GPU.')


In [ ]:
save_name = 'model1'  # change to whatever checkpoint you trained

ckpt = torch.load(save_name + '.pt', map_location=device, weights_only=False)
params = ckpt['params']
model = SindyAutoencoder(params).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def evaluate(data):
    batch = {k: torch.as_tensor(data[k], dtype=torch.float32, device=device)
             for k in ('x', 'dx', 'ddx') if k in data}
    with torch.no_grad():
        out = model(batch)
    return {k: v.detach().cpu().numpy() for k, v in out.items()}


In [ ]:
_, _, test_data = get_rd_data()
results = evaluate(test_data)

z_sim = sindy_simulate(results['z'][0], test_data['t'][:, 0],
                       params['coefficient_mask']*results['sindy_coefficients'],
                       params['poly_order'], params['include_sine'])

fig, axes = plt.subplots(2, 1, figsize=(4, 4))
axes[0].plot(results['z'][:, 0], color='#888888', linewidth=2)
axes[0].plot(z_sim[:, 0], '--', linewidth=2)
axes[1].plot(results['z'][:, 1], color='#888888', linewidth=2)
axes[1].plot(z_sim[:, 1], '--', linewidth=2)
plt.show()


In [ ]:
decoder_x_error = np.mean((test_data['x'] - results['x_decode'])**2)/np.mean(test_data['x']**2)
decoder_dx_error = np.mean((test_data['dx'] - results['dx_decode'])**2)/np.mean(test_data['dx']**2)
sindy_dz_error = np.mean((results['dz'] - results['dz_predict'])**2)/np.mean(results['dz']**2)

print('Decoder relative error: %f' % decoder_x_error)
print('Decoder relative SINDy error: %f' % decoder_dx_error)
print('SINDy relative error, z: %f' % sindy_dz_error)
